### Cell 1: Import libraries and set paths

Import pandas and NumPy and confirm the cleaned dataset exists at the fixed project path.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("E:/Quantum Project")
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "SP500_cleaned_eligible_tickers.csv"
)

print("Dataset found:", DATA_PATH.exists())

Dataset found: True


### Cell 2: Load cleaned prices

Read the cleaned prices, sort by ticker and date, and report rows and stocks.

In [2]:
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])

df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print("Rows:", len(df))
print("Stocks:", df["Ticker"].nunique())

df.head()

Rows: 2405879


Stocks: 372


,Ticker,Date,Open,High,Low,Close,Adj Close,Volume
0,A,2000-01-03,47.07,47.18,40.27,43.04,43.04,4674353
1,A,2000-01-04,40.72,41.17,38.70,39.75,39.75,4765083
2,A,2000-01-05,39.60,39.75,36.05,37.28,37.28,5758642
3,A,2000-01-06,36.83,37.06,34.74,35.86,35.86,2534434
4,A,2000-01-07,35.30,39.41,35.27,38.85,38.85,2819626


### Cell 3: Compare price columns

Count rows where Close and Adj Close differ to choose the price column.

In [3]:
different = ~np.isclose(df["Close"], df["Adj Close"])

print("Rows where Close and Adj Close differ:", different.sum())

df.loc[
    different,
    ["Ticker", "Date", "Close", "Adj Close"]
].head()

Rows where Close and Adj Close differ: 0


,Ticker,Date,Close,Adj Close


### Cell 4: Calculate one-observation returns

Compute each ticker's log return from the previous observation.

In [4]:
df["return_1d"] = df.groupby("Ticker")["Close"].transform(
    lambda prices: np.log(prices / prices.shift(1))
)

df[["Ticker", "Date", "Close", "return_1d"]].head(10)

,Ticker,Date,Close,return_1d
0,A,2000-01-03,43.04,NaN
1,A,2000-01-04,39.75,-0.079520
2,A,2000-01-05,37.28,-0.064153
3,A,2000-01-06,35.86,-0.038835
4,A,2000-01-07,38.85,0.080086
5,A,2000-01-10,41.21,0.058973
6,A,2000-01-11,40.65,-0.013682
7,A,2000-01-12,39.82,-0.020630
8,A,2000-01-13,40.42,0.014955
9,A,2000-01-14,40.87,0.011072


### Cell 5: Calculate longer returns

Compute 5-, 10- and 20-observation log returns per ticker.

In [5]:
for window in [5, 10, 20]:
    df[f"return_{window}d"] = df.groupby("Ticker")["Close"].transform(
        lambda prices: np.log(prices / prices.shift(window))
    )

df[
    ["Ticker", "Date", "Close",
     "return_1d", "return_5d", "return_10d", "return_20d"]
].head(25)

,Ticker,Date,Close,return_1d,return_5d,return_10d,return_20d
0,A,2000-01-03,43.04,NaN,NaN,NaN,NaN
1,A,2000-01-04,39.75,-0.079520,NaN,NaN,NaN
2,A,2000-01-05,37.28,-0.064153,NaN,NaN,NaN
3,A,2000-01-06,35.86,-0.038835,NaN,NaN,NaN
4,A,2000-01-07,38.85,0.080086,NaN,NaN,NaN
5,A,2000-01-10,41.21,0.058973,-0.043449,NaN,NaN
6,A,2000-01-11,40.65,-0.013682,0.022389,NaN,NaN
7,A,2000-01-12,39.82,-0.020630,0.065912,NaN,NaN
8,A,2000-01-13,40.42,0.014955,0.119702,NaN,NaN
9,A,2000-01-14,40.87,0.011072,0.050688,NaN,NaN


### Cell 6: Calculate rolling volatility

Compute the rolling standard deviation of one-observation returns over 5, 10 and 20 observations.

In [6]:
for window in [5, 10, 20]:
    df[f"volatility_{window}d"] = (
        df.groupby("Ticker")["return_1d"]
        .transform(
            lambda returns: returns.rolling(
                window=window,
                min_periods=window
            ).std()
        )
    )

df[
    ["Ticker", "Date", "return_1d",
     "volatility_5d", "volatility_10d", "volatility_20d"]
].head(25)

,Ticker,Date,return_1d,volatility_5d,volatility_10d,volatility_20d
0,A,2000-01-03,NaN,NaN,NaN,NaN
1,A,2000-01-04,-0.079520,NaN,NaN,NaN
2,A,2000-01-05,-0.064153,NaN,NaN,NaN
3,A,2000-01-06,-0.038835,NaN,NaN,NaN
4,A,2000-01-07,0.080086,NaN,NaN,NaN
5,A,2000-01-10,0.058973,0.073248,NaN,NaN
6,A,2000-01-11,-0.013682,0.062454,NaN,NaN
7,A,2000-01-12,-0.020630,0.052781,NaN,NaN
8,A,2000-01-13,0.014955,0.044335,NaN,NaN
9,A,2000-01-14,0.011072,0.031317,NaN,NaN


### Cell 7: Calculate moving-average distances

Compute how far the close is from its 5-, 10-, 20- and 50-observation moving averages.

In [7]:
for window in [5, 10, 20, 50]:
    moving_average = df.groupby("Ticker")["Close"].transform(
        lambda prices: prices.rolling(
            window=window,
            min_periods=window
        ).mean()
    )

    df[f"ma_distance_{window}d"] = df["Close"] / moving_average - 1

df[
    ["Ticker", "Date", "Close",
     "ma_distance_5d", "ma_distance_10d",
     "ma_distance_20d", "ma_distance_50d"]
].head(55)

,Ticker,Date,Close,ma_distance_5d,ma_distance_10d,ma_distance_20d,ma_distance_50d
0,A,2000-01-03,43.04,NaN,NaN,NaN,NaN
1,A,2000-01-04,39.75,NaN,NaN,NaN,NaN
2,A,2000-01-05,37.28,NaN,NaN,NaN,NaN
3,A,2000-01-06,35.86,NaN,NaN,NaN,NaN
4,A,2000-01-07,38.85,-0.002721,NaN,NaN,NaN
5,A,2000-01-10,41.21,0.067893,NaN,NaN,NaN
6,A,2000-01-11,40.65,0.048491,NaN,NaN,NaN
7,A,2000-01-12,39.82,0.013799,NaN,NaN,NaN
8,A,2000-01-13,40.42,0.005723,NaN,NaN,NaN
9,A,2000-01-14,40.87,0.006799,0.027530,NaN,NaN


### Cell 8: Calculate intraday price features

Compute the high-low range relative to the close and the open-to-close log return.

In [8]:
df["intraday_range"] = (df["High"] - df["Low"]) / df["Close"]

df["close_open_return"] = np.log(df["Close"] / df["Open"])

df[
    ["Ticker", "Date", "Open", "High", "Low", "Close",
     "intraday_range", "close_open_return"]
].head(10)

,Ticker,Date,Open,High,Low,Close,intraday_range,close_open_return
0,A,2000-01-03,47.07,47.18,40.27,43.04,0.160548,-0.089506
1,A,2000-01-04,40.72,41.17,38.70,39.75,0.062138,-0.024110
2,A,2000-01-05,39.60,39.75,36.05,37.28,0.099249,-0.060372
3,A,2000-01-06,36.83,37.06,34.74,35.86,0.064696,-0.026690
4,A,2000-01-07,35.30,39.41,35.27,38.85,0.106564,0.095825
5,A,2000-01-10,41.24,41.62,40.38,41.21,0.030090,-0.000728
6,A,2000-01-11,41.21,41.21,39.71,40.65,0.036900,-0.013682
7,A,2000-01-12,40.65,40.65,38.29,39.82,0.059267,-0.020630
8,A,2000-01-13,40.87,41.73,39.45,40.42,0.056408,-0.011072
9,A,2000-01-14,40.05,41.47,40.05,40.87,0.034744,0.020268


### Cell 9: Calculate volume features

Compute the log-volume change and a 20-observation volume z-score, treating zero volume as missing.

In [9]:
# Treat zero volume as unavailable for the logarithm.
log_volume = np.log(df["Volume"].where(df["Volume"] > 0))

# Change in log volume, calculated separately for each stock.
df["volume_log_change"] = log_volume.groupby(df["Ticker"]).diff()

# Compare today's volume with its recent 20-day history.
volume_group = df.groupby("Ticker")["Volume"]

volume_mean = volume_group.transform(
    lambda volume: volume.rolling(20, min_periods=20).mean()
)

volume_std = volume_group.transform(
    lambda volume: volume.rolling(20, min_periods=20).std()
)

df["volume_zscore_20d"] = (
    (df["Volume"] - volume_mean) / volume_std.replace(0, np.nan)
)

df[
    ["Ticker", "Date", "Volume",
     "volume_log_change", "volume_zscore_20d"]
].head(25)

,Ticker,Date,Volume,volume_log_change,volume_zscore_20d
0,A,2000-01-03,4674353,NaN,NaN
1,A,2000-01-04,4765083,0.019224,NaN
2,A,2000-01-05,5758642,0.189387,NaN
3,A,2000-01-06,2534434,-0.820731,NaN
4,A,2000-01-07,2819626,0.106634,NaN
5,A,2000-01-10,2148446,-0.271859,NaN
6,A,2000-01-11,1855985,-0.146329,NaN
7,A,2000-01-12,1429874,-0.260829,NaN
8,A,2000-01-13,1134337,-0.231538,NaN
9,A,2000-01-14,1316916,0.149244,NaN


### Cell 10: Calculate simple-rolling RSI

Compute a 14-observation RSI from simple rolling averages of gains and losses.

In [10]:
def calculate_rsi(prices, window=14):
    changes = prices.diff()

    average_gain = changes.clip(lower=0).rolling(
        window, min_periods=window
    ).mean()

    average_loss = (-changes.clip(upper=0)).rolling(
        window, min_periods=window
    ).mean()

    total_movement = average_gain + average_loss

    rsi = 100 * average_gain / total_movement.replace(0, np.nan)

    # A completely flat window receives a neutral value.
    return rsi.mask(total_movement == 0, 50)


df["rsi_14d"] = df.groupby("Ticker")["Close"].transform(calculate_rsi)

df[["Ticker", "Date", "Close", "rsi_14d"]].head(20)

,Ticker,Date,Close,rsi_14d
0,A,2000-01-03,43.04,NaN
1,A,2000-01-04,39.75,NaN
2,A,2000-01-05,37.28,NaN
3,A,2000-01-06,35.86,NaN
4,A,2000-01-07,38.85,NaN
5,A,2000-01-10,41.21,NaN
6,A,2000-01-11,40.65,NaN
7,A,2000-01-12,39.82,NaN
8,A,2000-01-13,40.42,NaN
9,A,2000-01-14,40.87,NaN


### Cell 11: Construct binary targets

Compute the next-observation log return and label it 1 if positive, otherwise 0; each ticker's last row stays missing.

In [11]:
next_close = df.groupby("Ticker")["Close"].shift(-1)

df["forward_return_1d"] = np.log(next_close / df["Close"])

df["target_up"] = (
    df["forward_return_1d"] > 0
).astype("Int64")

# Each stock's final row has no future price, so its label is unknown.
df.loc[df["forward_return_1d"].isna(), "target_up"] = pd.NA

df[
    ["Ticker", "Date", "Close", "forward_return_1d", "target_up"]
].head(10)

,Ticker,Date,Close,forward_return_1d,target_up
0,A,2000-01-03,43.04,-0.079520,0
1,A,2000-01-04,39.75,-0.064153,0
2,A,2000-01-05,37.28,-0.038835,0
3,A,2000-01-06,35.86,0.080086,1
4,A,2000-01-07,38.85,0.058973,1
5,A,2000-01-10,41.21,-0.013682,0
6,A,2000-01-11,40.65,-0.020630,0
7,A,2000-01-12,39.82,0.014955,1
8,A,2000-01-13,40.42,0.011072,1
9,A,2000-01-14,40.87,0.044739,1


### Cell 12: Define model input columns

List the 16 predictor columns, none of which uses future prices.

In [12]:
feature_columns = [
    "return_1d", "return_5d", "return_10d", "return_20d",
    "volatility_5d", "volatility_10d", "volatility_20d",
    "ma_distance_5d", "ma_distance_10d",
    "ma_distance_20d", "ma_distance_50d",
    "intraday_range", "close_open_return",
    "volume_log_change", "volume_zscore_20d",
    "rsi_14d",
]

print("Number of input features:", len(feature_columns))

missing_summary = (
    df[feature_columns + ["target_up"]]
    .isna()
    .sum()
    .rename("Missing rows")
)

missing_summary

Number of input features: 16


return_1d              372
return_5d             1860
return_10d            3720
return_20d            7440
volatility_5d         1860
volatility_10d        3720
volatility_20d        7440
ma_distance_5d        1488
ma_distance_10d       3348
ma_distance_20d       7068
ma_distance_50d      18228
intraday_range           0
close_open_return        0
volume_log_change      741
volume_zscore_20d     7068
rsi_14d               5208
target_up              372
Name: Missing rows, dtype: int64

### Cell 13: Inspect feature quality

Count missing and infinite values per feature and missing targets.

In [13]:
infinite_counts = pd.Series(
    np.isinf(df[feature_columns].to_numpy()).sum(axis=0),
    index=feature_columns,
    name="Infinite values"
)

quality_summary = pd.DataFrame({
    "Missing values": df[feature_columns].isna().sum(),
    "Infinite values": infinite_counts
})

display(quality_summary)

print("Missing targets:", df["target_up"].isna().sum())

,Missing values,Infinite values
return_1d,372,0
return_5d,1860,0
return_10d,3720,0
return_20d,7440,0
volatility_5d,1860,0
volatility_10d,3720,0
volatility_20d,7440,0
ma_distance_5d,1488,0
ma_distance_10d,3348,0
ma_distance_20d,7068,0


Missing targets: 372


### Cell 14: Filter incomplete observations

Treat infinite values as missing and drop rows with incomplete features or targets.

In [14]:
# Keep df unchanged for reference.
feature_df = df.copy()

# Treat infinite feature values as missing.
feature_df[feature_columns] = feature_df[feature_columns].replace(
    [np.inf, -np.inf], np.nan
)

rows_before = len(feature_df)

feature_df = feature_df.dropna(
    subset=feature_columns + ["target_up"]
).copy()

feature_df["target_up"] = feature_df["target_up"].astype(int)

print("Rows before cleaning:", rows_before)
print("Rows removed:", rows_before - len(feature_df))
print("Rows remaining:", len(feature_df))
print("Stocks remaining:", feature_df["Ticker"].nunique())

Rows before cleaning: 2405879
Rows removed: 18954
Rows remaining: 2386925
Stocks remaining: 372


### Cell 15: Summarize binary classes

Show the count and percentage of UP and DOWN-or-unchanged labels.

In [15]:
class_summary = (
    feature_df["target_up"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis("Target")
    .to_frame("Rows")
)

class_summary["Percentage"] = (
    100 * class_summary["Rows"] / len(feature_df)
).round(2)

class_summary["Meaning"] = ["DOWN or unchanged", "UP"]

class_summary

,Rows,Percentage,Meaning
Target,,,
0,1171449,49.08,DOWN or unchanged
1,1215476,50.92,UP


### Cell 16: Validate model inputs

Assert there are no missing or infinite features, targets are 0 or 1, and ticker-date keys are unique.

In [16]:
assert not feature_df[feature_columns].isna().any().any()
assert np.isfinite(feature_df[feature_columns].to_numpy()).all()
assert feature_df["target_up"].isin([0, 1]).all()
assert not feature_df.duplicated(["Ticker", "Date"]).any()

print("All four checks passed.")

All four checks passed.


### Cell 17: Save binary feature dataset

Write identifiers, the 16 features and target_up to SP500_features.csv, and the feature list to feature_columns.json.

In [17]:
import json

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_PATH = OUTPUT_DIR / "SP500_features.csv"
COLUMNS_PATH = OUTPUT_DIR / "feature_columns.json"

# Save identifiers, the 16 input features, and the target.
columns_to_save = ["Ticker", "Date"] + feature_columns + ["target_up"]

feature_df[columns_to_save].to_csv(FEATURE_PATH, index=False)

COLUMNS_PATH.write_text(
    json.dumps(feature_columns, indent=2),
    encoding="utf-8"
)

print("Dataset saved:", FEATURE_PATH)
print("Feature list saved:", COLUMNS_PATH)

Dataset saved: E:\Quantum Project\data\processed\SP500_features.csv
Feature list saved: E:\Quantum Project\data\processed\feature_columns.json


### Cell 18: Verify saved feature schema

Reload the saved files and confirm the saved columns and feature list match.

In [18]:
saved_sample = pd.read_csv(FEATURE_PATH, nrows=5)
saved_features = json.loads(COLUMNS_PATH.read_text(encoding="utf-8"))

assert saved_features == feature_columns
assert saved_sample.columns.tolist() == columns_to_save

print("Saved input features:", len(saved_features))
print("Saved columns:", len(saved_sample.columns))
print(f"CSV size: {FEATURE_PATH.stat().st_size / (1024 ** 2):.1f} MB")

saved_sample

Saved input features: 16
Saved columns: 19
CSV size: 790.5 MB


,Ticker,Date,return_1d,return_5d,return_10d,return_20d,volatility_5d,volatility_10d,volatility_20d,ma_distance_5d,ma_distance_10d,ma_distance_20d,ma_distance_50d,intraday_range,close_open_return,volume_log_change,volume_zscore_20d,rsi_14d,target_up
0,A,2000-03-14,-0.078185,-0.156963,0.155691,0.462397,0.066357,0.144109,0.111260,-0.132551,-0.082598,0.061472,0.378591,0.141144,-0.091481,0.293572,0.530359,57.902825,0
1,A,2000-03-15,-0.060513,-0.285446,0.045844,0.333168,0.036353,0.145430,0.112221,-0.136924,-0.139794,-0.014802,0.285309,0.130180,-0.114573,-0.145393,0.221896,52.659701,1
2,A,2000-03-16,0.032842,-0.259187,0.114423,0.376692,0.047911,0.144933,0.112085,-0.058414,-0.119537,0.002080,0.312989,0.093099,0.017583,-0.095214,0.031607,53.466387,1
3,A,2000-03-17,0.025601,-0.159032,0.114766,0.222108,0.056250,0.144937,0.105510,-0.000718,-0.105466,0.017647,0.329660,0.073204,0.015030,-0.571574,-0.710450,57.225225,0
4,A,2000-03-20,-0.062846,-0.143101,-0.334937,0.193295,0.053298,0.060964,0.106352,-0.033684,-0.130907,-0.052299,0.234104,0.074717,-0.057167,-0.096008,-0.719308,53.432627,1


### Cell 19: Export separate three-class targets

Write SP500_targets.csv with simple forward returns, label end dates and fixed ±0.25% three-class labels, kept out of the feature file.

In [19]:
import runpy
extension = runpy.run_path(str(PROJECT_ROOT / 'experiments/multiclass_extension.py'))
target_export = feature_df[['Ticker', 'Date', 'target_up']].copy()
# Use original next-observation prices, not shifts after feature-row filtering.
target_export['forward_return_simple'] = np.expm1(feature_df['forward_return_1d'])
target_export['label_end'] = df.groupby('Ticker')['Date'].shift(-1).loc[feature_df.index]
target_export['target_three'] = extension['labels'](target_export['forward_return_simple']).astype(int)
assert target_export.label_end.notna().all()
assert target_export.target_three.isin([0, 1, 2]).all()
assert np.array_equal(target_export.target_up, (target_export.forward_return_simple > 0).astype(int))
TARGET_PATH = OUTPUT_DIR / 'SP500_targets.csv'
target_export.to_csv(TARGET_PATH, index=False)
display(target_export.target_three.value_counts().sort_index().rename(index={0:'DOWN',1:'NEUTRAL',2:'UP'}).to_frame('Rows'))
print('Target-only metadata saved:', TARGET_PATH)
print('Threshold: fixed ±0.25% simple return; endpoints are NEUTRAL. No threshold optimization.')

,Rows
target_three,
DOWN,964744
NEUTRAL,369698
UP,1052483


Target-only metadata saved: E:\Quantum Project\data\processed\SP500_targets.csv
Threshold: fixed ±0.25% simple return; endpoints are NEUTRAL. No threshold optimization.
